# Lab 4 · ความถี่ของคลื่นและ noise ที่สถานี CHTO
รันทีละ cell ตามการ์ด Lab บนเว็บ (Shift+Enter)

## Cell 1

In [ ]:
!pip -q install obspy

## Cell 2

In [ ]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from obspy.signal.spectral_estimation import get_nlnm, get_nhnm
from scipy.signal import welch
import numpy as np, matplotlib.pyplot as plt
client = Client("IRIS")

def psd(start, minutes=60, loc="00"):
    t = UTCDateTime(start)
    st = client.get_waveforms("IU", "CHTO", loc, "BHZ", t, t + minutes*60, attach_response=True)
    st.merge(fill_value=0)
    tr = st[0]
    tr.remove_response(output="ACC", pre_filt=(0.002, 0.004, 15, 18))
    f, p = welch(tr.data, fs=tr.stats.sampling_rate, nperseg=2**13)
    return f[1:], 10*np.log10(p[1:]), tr

def at(fq, f, db):
    return round(float(db[np.argmin(abs(f - fq))]))

f_n, night, tr = psd("2025-03-27T18:00:00")   # 01:00 น. เวลาไทย
print(tr.id, tr.stats.sampling_rate, "Hz,", tr.stats.npts, "samples")

## Cell 3

In [ ]:
Tl, Nl = get_nlnm(); Th, Nh = get_nhnm()
lo = lambda fq: round(float(np.interp(1/fq, Tl[::-1], Nl[::-1])))
hi = lambda fq: round(float(np.interp(1/fq, Th[::-1], Nh[::-1])))
for fq in [0.01, 0.2, 1, 10]:
    print(f"{fq:>5} Hz  CHTO {at(fq, f_n, night)} dB   NLNM {lo(fq)}   NHNM {hi(fq)}")
band = (f_n > 0.05) & (f_n < 1)
fpk = f_n[band][np.argmax(night[band])]
print(f"microseism peak {fpk:.2f} Hz = period {1/fpk:.1f} s")

## Cell 4

In [ ]:
f_d, day, _ = psd("2025-03-27T06:00:00")      # 13:00 น. เวลาไทย
for fq in [0.2, 1, 5, 10]:
    print(f"{fq:>5} Hz  night {at(fq, f_n, night)}  day {at(fq, f_d, day)}  diff {at(fq, f_d, day) - at(fq, f_n, night):+d} dB")

## Cell 5

In [ ]:
f_q, quake, trq = psd("2025-03-28T06:21:30", minutes=10, loc="10")
print(f"max vertical acceleration {abs(trq.data).max():.3f} m/s² = {abs(trq.data).max()/9.81:.4f} g")
for fq in [0.1, 1, 5]:
    print(f"{fq:>5} Hz  quake {at(fq, f_q, quake)}  night {at(fq, f_n, night)}  SNR {at(fq, f_q, quake) - at(fq, f_n, night):+d} dB")

## Cell 6

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(f_n, night, label="night 01:00")
ax.semilogx(f_d, day, label="day 13:00")
ax.semilogx(f_q, quake, label="Mandalay M7.7, 10 min")
ax.semilogx(1/Tl, Nl, "k--", lw=1, label="NLNM / NHNM")
ax.semilogx(1/Th, Nh, "k--", lw=1)
ax.set(xlim=(0.005, 18), ylim=(-200, -30), xlabel="Frequency (Hz)",
       ylabel="PSD (dB rel. 1 (m/s²)²/Hz)", title="IU.CHTO BHZ")
ax.grid(True, which="both", alpha=.3); ax.legend()
plt.savefig("chto_psd.png", dpi=150, bbox_inches="tight")